# Actividad 12 — Etapa A: Consolidación y Verificación del Pipeline Modular
### Proyecto Integrador — Procesamiento de Grandes Bases de Datos
**Flujo:** `Extract → Validate (Input) → Transform & Clean → Validate (Output) → Load → Verify`

---
## 1. Contexto, Arquitectura Modular (`src/`) y Decisiones Técnicas
En esta etapa consolidamos el pipeline modular desarrollado en PySpark bajo el estándar de la **Clase 11 y Actividad 11**, verificando que genere de forma reproducible y sin errores la **Capa Gold** en `data/processed/proyecto_integrado_mensual.parquet`.

### Responsabilidades de los módulos en `src/`
* **`src/ingestion/extract.py`:** Encargado de la ingesta de las tres fuentes mediante esquemas explícitos (`StructType`) para los CSV de Banxico (`tipo_cambio_fix.csv`) y Yahoo Finance (`s&p500_mensual.csv`), así como la lectura del Parquet base del Laboratorio 8 (`finanzas_igae_mensual_final.parquet`). Proteger el contrato de tipos en la ingesta evita inferencias erróneas en fechas o números con texto.
* **`src/quality/validate.py`:** Implementa contratos de datos estrictos en puntos de control estratégicos:
  * `validate_lab8_input`: Comprueba existencia de 8 columnas base, verifica tipos numéricos y exige unicidad de periodos (0 duplicados).
  * `validate_tc_input` y `validate_sp500_input`: Verifican que los CSV contengan las columnas requeridas (`fecha`, `dato`, `sp500_cierre`, `sp500_promedio`, `vix_cierre`).
  * `validate_prepared_monthly`: Asegura que las series agregadas tengan exactamente 1 fila por `PERIODO` antes de realizar los *joins*.
  * `validate_output`: Audita el esquema final (12 columnas), igualdad de cardinalidad (187 filas) y garantiza **0 valores nulos**.
* **`src/transformations/integrate_enrich.py`:** Contiene la lógica de transformación, limpieza e integración:
  * `prepare_tipo_cambio`: Filtra registros nulos y cadenas `"N/E"`, convierte fechas de `dd/MM/yyyy` a `date`, castea a `double`, trunca a mes (`PERIODO`) y toma el último valor del mes mediante `F.max_by("tc_valor", "fecha_dt").alias("TC_CIERRE")`.
  * `prepare_sp500`: Trunca la fecha al inicio de mes (`PERIODO`) y proyecta `SP500_CIERRE`, `SP500_PROMEDIO` y `VIX_CIERRE`.
  * `integrate_and_enrich`: Ejecuta `Inner Join` sobre `PERIODO` envolviendo las tablas financieras con `F.broadcast()` para forzar un `BroadcastHashJoin`, eliminando intercambios de red innecesarios.
* **`pipeline.py`:** Orquestador principal en la raíz del proyecto. Coordina las etapas, asigna etiquetas de observabilidad en Spark UI con `setJobGroup("PROYECTO_PIPELINE", ...)` y registra la evidencia en `reports/ultima_ejecucion.json`.

## 2. Preparación del Entorno y Sesión Spark
Garantizamos que el entorno apunte a la raíz del proyecto para importar `src/` e inicializamos la sesión local de PySpark.

In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pipeline.py").exists():
    os.chdir("..")

print("Directorio de trabajo:", Path.cwd())
assert Path("pipeline.py").exists(), "pipeline.py debe residir en la raíz"


Directorio de trabajo: c:\Users\julia\OneDrive\Escritorio\MCD\Procesamiento de GBD\Proyecto_Integrador_PGBD


In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Actividad12_EtapaA_ConsolidacionPipeline")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

print("Versión de PySpark:", spark.version)
print("Spark UI disponible en:", spark.sparkContext.uiWebUrl)


Versión de PySpark: 3.5.9
Spark UI disponible en: http://Fanta:4040


## 3. Extracción de Fuentes (`src/ingestion/extract.py`)
Reconocimiento y carga distribuida de las tres fuentes en `data/raw/`.

In [3]:
from pipeline import LAB8_PATH, TC_PATH, SP500_PATH, OUTPUT_PATH, REPORT_PATH, run_pipeline
from src.ingestion.extract import extract_lab8, extract_tipo_cambio, extract_sp500

df_lab8 = extract_lab8(spark, LAB8_PATH)
df_tc_raw = extract_tipo_cambio(spark, TC_PATH)
df_sp_raw = extract_sp500(spark, SP500_PATH)

print(f"1. Lab 8 (SHCP + INPC + IGAE): {df_lab8.count()} registros")
df_lab8.select("PERIODO", "INGRESOS_REALES", "GASTOS_REALES", "INPC", "Total").show(3, truncate=False)

print(f"2. Banxico Tipo de Cambio (Diario): {df_tc_raw.count()} registros")
df_tc_raw.show(3, truncate=False)

print(f"3. S&P 500 + VIX (Mensual): {df_sp_raw.count()} registros")
df_sp_raw.select("fecha", "sp500_cierre", "sp500_promedio", "vix_cierre").show(3, truncate=False)


[EXTRACT] Leyendo Parquet Lab 8: data/raw/finanzas_igae_mensual_final.parquet
[EXTRACT] Leyendo CSV Tipo de Cambio: data/raw/tipo_cambio_fix.csv
[EXTRACT] Leyendo CSV S&P 500 + VIX: data/raw/s&p500_mensual.csv
1. Lab 8 (SHCP + INPC + IGAE): 187 registros
+----------+--------------------+--------------------+---------------+----------------+
|PERIODO   |INGRESOS_REALES     |GASTOS_REALES       |INPC           |Total           |
+----------+--------------------+--------------------+---------------+----------------+
|2016-03-01|5.265423178640631E8 |4.1988668213510334E8|89.910000600998|94.1272843004329|
|2019-11-01|4.865511981784747E8 |4.681168725894728E8 |105.346        |102.895473729605|
|2018-09-01|4.0800993548266786E8|4.8907894043595624E8|100.917        |97.9977567165606|
+----------+--------------------+--------------------+---------------+----------------+
only showing top 3 rows

2. Banxico Tipo de Cambio (Diario): 8791 registros
+----------+------+
|fecha     |dato  |
+----------+-

## 4. Validación de Entrada y Limpieza (`src/quality/validate.py` y `src/transformations/`)
Ejecutamos las validaciones contractuales y la agregación mensual del tipo de cambio y del S&P 500.

In [4]:
from src.quality.validate import (
    validate_lab8_input,
    validate_tc_input,
    validate_sp500_input,
    validate_prepared_monthly,
)
from src.transformations.integrate_enrich import prepare_tipo_cambio, prepare_sp500, integrate_and_enrich

n_lab8 = validate_lab8_input(df_lab8)
n_tc = validate_tc_input(df_tc_raw)
n_sp = validate_sp500_input(df_sp_raw)
print(f"Validación exitosa -> Lab8: {n_lab8} | TC: {n_tc} | S&P 500: {n_sp}")

df_tc_m = prepare_tipo_cambio(df_tc_raw)
df_sp_m = prepare_sp500(df_sp_raw)

validate_prepared_monthly(df_tc_m, "TC_MENSUAL")
validate_prepared_monthly(df_sp_m, "SP500_MENSUAL")

print("Meses únicos listos para integración 1:1:")
print(f"  TC mensualizado: {df_tc_m.count()} meses | S&P500 mensualizado: {df_sp_m.count()} meses")


[VALIDATE] Revisando entrada Lab 8
[VALIDATE] Revisando entrada Tipo de Cambio
[VALIDATE] Revisando entrada S&P 500
Validación exitosa -> Lab8: 187 | TC: 8791 | S&P 500: 442
[TRANSFORM] Preparando Tipo de Cambio (USD/MXN - solo cierre mensual)
[TRANSFORM] Homologando PERIODO en S&P 500 y VIX
Meses únicos listos para integración 1:1:
  TC mensualizado: 421 meses | S&P500 mensualizado: 442 meses


## 5. Integración 1:1 y Auditoría del Plan Físico (`explain formatted`)
Inspeccionamos cómo Spark compila la consulta, confirmando `BroadcastHashJoin` para evitar shuffle.

In [5]:
df_integrado = integrate_and_enrich(df_lab8, df_tc_m, df_sp_m)

print("=== PLAN FÍSICO COMPILED (explain formatted) ===")
df_integrado.explain("formatted")

print("=== ESQUEMA DE SALIDA (12 COLUMNAS) ===")
df_integrado.printSchema()
df_integrado.show(5, truncate=False)


[TRANSFORM] Integrando Lab 8 + TC_CIERRE + SP500_CIERRE (Inner Join 1:1)
=== PLAN FÍSICO COMPILED (explain formatted) ===
== Physical Plan ==
* Sort (21)
+- Exchange (20)
   +- * Project (19)
      +- * BroadcastHashJoin Inner BuildRight (18)
         :- * Project (13)
         :  +- * BroadcastHashJoin Inner BuildRight (12)
         :     :- * Filter (3)
         :     :  +- * ColumnarToRow (2)
         :     :     +- Scan parquet  (1)
         :     +- BroadcastExchange (11)
         :        +- * HashAggregate (10)
         :           +- Exchange (9)
         :              +- * HashAggregate (8)
         :                 +- * Project (7)
         :                    +- * Project (6)
         :                       +- * Filter (5)
         :                          +- Scan csv  (4)
         +- BroadcastExchange (17)
            +- * Project (16)
               +- * Filter (15)
                  +- Scan csv  (14)


(1) Scan parquet 
Output [8]: [PERIODO#0, INGRESOS_REALES#1, GAS

## 6. Prueba Negativa de Calidad
Demostramos que los contratos en `validate.py` detienen el flujo ante anomalías (duplicados simulados).

In [6]:
df_duplicado = df_lab8.unionByName(df_lab8.limit(1))
try:
    validate_lab8_input(df_duplicado)
    raise AssertionError("El validador debió fallar por duplicado")
except ValueError as exc:
    print("Prueba negativa exitosa. El contrato interceptó el error:", exc)


[VALIDATE] Revisando entrada Lab 8
Prueba negativa exitosa. El contrato interceptó el error: [LAB8] Existen 1 periodos duplicados en Lab 8.


## 7. Ejecución de Extremo a Extremo con `pipeline.py` y Verificación en Disco
Ejecutamos el orquestador y revisamos el reporte JSON y el archivo Parquet persistido.

In [7]:
import json

df_resultado = run_pipeline(spark)

print("\n=== REPORTE DE EJECUCIÓN (reports/ultima_ejecucion.json) ===")
reporte = json.loads(Path(REPORT_PATH).read_text(encoding="utf-8"))
print(json.dumps(reporte, indent=2))

# Auditoría de persistencia
df_disco = spark.read.parquet(OUTPUT_PATH)
print(f"\nArchivo validado en disco: {OUTPUT_PATH}")
print(f"Dimensiones: {df_disco.count()} filas x {len(df_disco.columns)} columnas")

nulos_total = df_disco.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_disco.columns])
print("Auditoría de valores nulos (debe ser 0 en todas las columnas):")
print(nulos_total.first().asDict())


=== INICIO DEL PIPELINE ===
[EXTRACT] Leyendo Parquet Lab 8: data/raw/finanzas_igae_mensual_final.parquet
[EXTRACT] Leyendo CSV Tipo de Cambio: data/raw/tipo_cambio_fix.csv
[EXTRACT] Leyendo CSV S&P 500 + VIX: data/raw/s&p500_mensual.csv
[VALIDATE] Revisando entrada Lab 8
[VALIDATE] Revisando entrada Tipo de Cambio
[VALIDATE] Revisando entrada S&P 500
[VALIDATE] Entradas correctas: {'lab8': 187, 'tipo_cambio_diario': 8791, 'sp500_mensual': 442}
[TRANSFORM] Preparando Tipo de Cambio (USD/MXN - solo cierre mensual)
[TRANSFORM] Homologando PERIODO en S&P 500 y VIX
[TRANSFORM] Integrando Lab 8 + TC_CIERRE + SP500_CIERRE (Inner Join 1:1)
[VALIDATE] Revisando salida integrada
[VALIDATE] Salida correcta: 187 registros
[LOAD] Guardando resultado Parquet en data/processed/proyecto_integrado_mensual.parquet
[VERIFY] Releyendo Parquet generado para confirmar integridad
[VERIFY] Relectura exitosa: 187 filas, 12 columnas
[OK] Resultado publicado en: data/processed/proyecto_integrado_mensual.parquet